In [1]:
# ============================================================
# Q2 — Mean Imputation vs MICE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

# IterativeImputer is experimental in some sklearn versions
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_squared_error


# ------------------------------------------------------------
# 1. Load California Housing dataset
# ------------------------------------------------------------



In [2]:

housing = fetch_california_housing(as_frame=True)

X = housing.data.copy()
y = housing.target.copy()

print("Original shape:", X.shape)

display(X.head())


Original shape: (20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
# ------------------------------------------------------------
# 2. Confirm original dataset has no missing values
# ------------------------------------------------------------

print("\nOriginal missing values:")
print(X.isnull().sum().sum())


# ------------------------------------------------------------
# 3. Artificially introduce 15% missing values
# ------------------------------------------------------------

rng = np.random.RandomState(42)

X_missing = X.copy()

missing_rate = 0.15

mask = rng.rand(*X_missing.shape) < missing_rate

X_missing[mask] = np.nan


Original missing values:
0


In [4]:
# ------------------------------------------------------------
# 4. Check missingness
# ------------------------------------------------------------

print("\nMissing percentage after injection:")

missing_percentage = (
    X_missing.isnull().mean() * 100
)

print(missing_percentage)


# ------------------------------------------------------------
# 5. Split data BEFORE fitting imputers
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_missing,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Missing percentage after injection:
MedInc        14.854651
HouseAge      15.169574
AveRooms      15.285853
AveBedrms     14.941860
Population    14.966085
AveOccup      14.874031
Latitude      14.622093
Longitude     15.237403
dtype: float64

Training shape: (16512, 8)
Testing shape: (4128, 8)


In [5]:

# ============================================================
# MODEL 1 — MEAN IMPUTATION
# ============================================================

mean_pipeline = Pipeline([
    
    (
        "imputer",
        SimpleImputer(strategy="mean")
    ),
    
    (
        "model",
        LinearRegression()
    )
])


In [6]:
# ------------------------------------------------------------
# 6. Train mean-imputation model
# ------------------------------------------------------------

mean_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 7. Predict
# ------------------------------------------------------------

mean_predictions = mean_pipeline.predict(
    X_test
)


In [7]:
# ------------------------------------------------------------
# 7. Predict
# ------------------------------------------------------------

mean_predictions = mean_pipeline.predict(
    X_test
)


# ------------------------------------------------------------
# 8. Calculate RMSE
# ------------------------------------------------------------

mean_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        mean_predictions
    )
)

print("\nMean Imputation RMSE:")
print(mean_rmse)




Mean Imputation RMSE:
0.8574044631500636


In [8]:
# ============================================================
# MODEL 2 — MICE / ITERATIVE IMPUTATION
# ============================================================

mice_pipeline = Pipeline([
    
    (
        "imputer",
        IterativeImputer(
            max_iter=10,
            random_state=42
        )
    ),
    
    (
        "model",
        LinearRegression()
    )
])


# ------------------------------------------------------------
# 9. Train MICE model
# ------------------------------------------------------------

mice_pipeline.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 10. Predict
# ------------------------------------------------------------

mice_predictions = mice_pipeline.predict(
    X_test
)

# ------------------------------------------------------------
# 11. Calculate RMSE
# ------------------------------------------------------------

mice_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        mice_predictions
    )
)

print("\nMICE RMSE:")
print(mice_rmse)



MICE RMSE:
0.800852640159686


In [9]:
# ============================================================
# 12. FINAL COMPARISON
# ============================================================

comparison_q2 = pd.DataFrame({
    
    "Imputation Method": [
        "Mean",
        "MICE / IterativeImputer"
    ],
    
    "RMSE": [
        mean_rmse,
        mice_rmse
    ]
})

print("\nFinal Comparison:")
display(comparison_q2)


# ------------------------------------------------------------
# 13. Difference
# ------------------------------------------------------------

difference = mean_rmse - mice_rmse

print("\nRMSE Difference (Mean - MICE):")
print(difference)


Final Comparison:


,Imputation Method,RMSE
0,Mean,0.857404
1,MICE / IterativeImputer,0.800853



RMSE Difference (Mean - MICE):
0.05655182299037753
